## Genetic Algorithm for the Travelling Salesman Problem

In [ ]:
# code adapted from:
# https://towardsdatascience.com/evolution-of-a-salesman-a-complete-genetic-algorithm-tutorial-for-python-6fe5d2b3ca35

# libraries used
import numpy as np, random, operator, pandas as pd, matplotlib.pyplot as plt

In [ ]:
# cities are nodes of the graph and represent genes
# (x, y) is the position of a city on the map

class City:

    # initialize a city as a class
    def __init__(self, x, y):
        self.x = x
        self.y = y

    # distance between two cities using the Pythagorean theorem
    def distance(self, city):
        xDis = abs(self.x - city.x)
        yDis = abs(self.y - city.y)
        distance = np.sqrt((xDis ** 2) + (yDis ** 2))
        return distance


    def __repr__(self):
        return "(" + str(self.x) + "," + str(self.y) + ")"


In [ ]:
# objective function
# the goal is to minimize the total edge weight of the cycle, i.e. the route length
# this is a constrained optimization problem
# constraint: the start and end of the route is the same node

class Fitness:
    def __init__(self, route):
        self.route = route
        self.distance = 0
        self.fitness = 0.0

    def routeDistance(self):
        if self.distance == 0:
            pathDistance = 0
            for i in range(0, len(self.route)):
                fromCity = self.route[i]
                toCity = None
                # if we are not at the last city, move to the next one,
                # otherwise close the cycle by returning to the first city
                if i + 1 < len(self.route):
                    toCity = self.route[i + 1]
                else:
                    toCity = self.route[0]
                pathDistance += fromCity.distance(toCity)
            self.distance = pathDistance
        return self.distance

    # we are looking for the route with minimal total weight
    # the fitness function maximizes 1 / route length
    def routeFitness(self):
        if self.fitness == 0:
            self.fitness = 1 / float(self.routeDistance())
        return self.fitness

## Creating a Random Hamiltonian Cycle

In [ ]:
# creates one initial random route which is then optimized

def createRoute(cityList):
    route = random.sample(cityList, len(cityList))
    return route

In [ ]:
# creating the initial population

def initialPopulation(popSize, cityList):
    population = []

    for i in range(0, popSize):
        population.append(createRoute(cityList))
    return population

## The Iterative Algorithm

In [ ]:
# evaluation of the objective function
# population     - input: a population of Hamiltonian cycles
# fitnessResults - result of the objective (fitness) function
# returns a list of routes sorted by fitness (1 / distance) in descending order

def rankRoutes(population):
    fitnessResults = {}
    for i in range(0, len(population)):
        fitnessResults[i] = Fitness(population[i]).routeFitness()

    #print(fitnessResults)
    return sorted(fitnessResults.items(), key = operator.itemgetter(1), reverse = True)

In [ ]:
# elitist selection: the best individuals survive into the next generation
# from the sorted list of routes we pick those with the lowest weight w, i.e. max(1/w)

def selection(popRanked, eliteSize):

    selectionResults = []
    df = pd.DataFrame(np.array(popRanked), columns=["Index","Fitness"])
    #print(df)
    df['cum_sum'] = df.Fitness.cumsum()
    #print(df)
    df['cum_perc'] = 100*df.cum_sum/df.Fitness.sum()
    #print(df)

    # routes selected for the next generation
    for i in range(0, eliteSize):
        selectionResults.append(popRanked[i][0])
    for i in range(0, len(popRanked) - eliteSize):
        pick = 100*random.random()
        # draw a random number from the uniform distribution on (0, 100)
        for i in range(0, len(popRanked)):
            # if the drawn number is less than or equal to the cumulative percentile, select that route
            # flipping the sign to > would select the maximal Hamiltonian cycle instead
            if pick <= df.iat[i,3]:
                selectionResults.append(popRanked[i][0])
                break
        #print(selectionResults)
    return selectionResults

In [ ]:
# mating pool - the input is the previous population
# selected routes are referenced by their identification number

def matingPool(population, selectionResults):
    matingpool = []
    for i in range(0, len(selectionResults)):
        index = selectionResults[i]
        matingpool.append(population[index])


    #print(matingpool)
    return matingpool

In [ ]:
# ordered crossover: a random segment is taken from parent1
# and the remaining cities are filled in from parent2 in their original order

def breed(parent1, parent2):
    child = []
    childP1 = []
    childP2 = []

    geneA = int(random.random() * len(parent1))
    geneB = int(random.random() * len(parent1))

    startGene = min(geneA, geneB)
    endGene = max(geneA, geneB)

    for i in range(startGene, endGene):
        childP1.append(parent1[i])

    childP2 = [item for item in parent2 if item not in childP1]

    child = childP1 + childP2
    return child

In [ ]:
def breedPopulation(matingpool, eliteSize):
    children = []
    length = len(matingpool) - eliteSize
    pool = random.sample(matingpool, len(matingpool))

    for i in range(0, eliteSize):
        children.append(matingpool[i])

    #print(children)

    for i in range(0, length):
        child = breed(pool[i], pool[len(matingpool)-i-1])
        children.append(child)
    return children

In [ ]:
# swap the positions of two cities with probability mutationRate
def mutate(individual, mutationRate):
    for swapped in range(len(individual)):
        if(random.random() < mutationRate):
            swapWith = int(random.random() * len(individual))

            city1 = individual[swapped]
            city2 = individual[swapWith]

            individual[swapped] = city2
            individual[swapWith] = city1
    return individual

In [ ]:
def mutatePopulation(population, mutationRate):
    mutatedPop = []

    for ind in range(0, len(population)):
        mutatedInd = mutate(population[ind], mutationRate)
        mutatedPop.append(mutatedInd)
    return mutatedPop

In [ ]:
def nextGeneration(currentGen, eliteSize, mutationRate):

    popRanked = rankRoutes(currentGen)

    # currentGen - input population of Hamiltonian cycles
    # popRanked  - list of routes sorted by fitness

    selectionResults = selection(popRanked, eliteSize)

    # selects routes from the original population with the lowest weight
    # plus a random number of additional routes via roulette-wheel selection

    matingpool = matingPool(currentGen, selectionResults)

    # selected routes are referenced by their identification number
    # this creates the population of parents

    children = breedPopulation(matingpool, eliteSize)

    # creates the population of children using replication, crossover and mutation

    nextGeneration = mutatePopulation(children, mutationRate)
    return nextGeneration

In [ ]:
def geneticAlgorithm(population, popSize, eliteSize, mutationRate, generations):
    pop = initialPopulation(popSize, population)
    print("Initial distance: " + str(1 / rankRoutes(pop)[0][1]))

    # distance of the best route in the random initial population

    for i in range(0, generations):
        pop = nextGeneration(pop, eliteSize, mutationRate)

    # pop          - previous population of Hamiltonian cycles
    # eliteSize    - number of routes we want to replicate unchanged
    # mutationRate - probability that two cities swap positions

    print("Final distance: " + str(1 / rankRoutes(pop)[0][1]))
    bestRouteIndex = rankRoutes(pop)[0][0]
    bestRoute = pop[bestRouteIndex]
    return bestRoute

## Graph Definition

In [ ]:
cityList = []

# a random graph of cities can be generated each time:
#for i in range(0,25):
#   cityList.append(City(x=int(random.random() * 200), y=int(random.random() * 200)))

# one fixed graph
# cityList is the list of cities and their positions
# 25 cities are chosen:
cityList = [City(13,93),City(84,162),City(119,56),City(122,117),City(82,4),City(168,40),City(121,106), \
            City(141,164),City(34,13),City(163,175),City(70,89),City(130,67),City(59,96),City(23,174), \
            City(186,174),City(143,134),City(72,152),City(100,111),City(152,101),City(109,77),City(27,162), \
            City(41,106),City(184,28),City(8,193),City(146,70)]

## Running the Algorithm

In [ ]:
ham_cycle = geneticAlgorithm(population=cityList, popSize=50, eliteSize=10, mutationRate=0.01, generations=500)

In [ ]:
def geneticAlgorithmPlot(population, popSize, eliteSize, mutationRate, generations):
    pop = initialPopulation(popSize, population)
    progress = []
    progress.append(1 / rankRoutes(pop)[0][1])

    for i in range(0, generations):
        pop = nextGeneration(pop, eliteSize, mutationRate)
        progress.append(1 / rankRoutes(pop)[0][1])

    plt.plot(progress)
    plt.title("Convergence to the shortest Hamiltonian cycle")
    plt.ylabel('distance')
    plt.xlabel('generation')
    plt.savefig('konvergence.png', dpi = 200)
    plt.show()

In [ ]:
geneticAlgorithmPlot(population=cityList, popSize=50, eliteSize=10, mutationRate=0.01, generations=500)

In [ ]:
# extract (x, y) coordinates of the resulting route for plotting
numeric = pd.DataFrame(ham_cycle, columns=["data"])
numeric['data'] = numeric['data'].astype(str)
numeric[['x', 'y_pred']] = numeric.data.str.split(",", expand = True)
numeric['x'] = numeric['x'].str[1:]
numeric[['y', ")"]] = numeric.y_pred.str.split(")", expand = True)

numeric['x'] = numeric['x'].astype(int)
numeric['y'] = numeric['y'].astype(int)
numeric.drop(columns=["data", "y_pred",")"])

In [ ]:
plt.scatter(numeric.x, numeric.y)
plt.plot(numeric.x, numeric.y, 'r-')

# connect the last city back to the first one to close the cycle
last_road = numeric.drop(labels=range(1, len(numeric.x)-1), axis=0)
plt.plot(last_road.x, last_road.y, 'r-')
plt.title("Hamiltonian cycle through all cities")
plt.xlabel("x")
plt.ylabel("y")
plt.savefig('ham.png', dpi = 200)